In [1]:
"""
=============================================================================
AUTOPILOT VQA — Stage 0 v2: Industrial Frame Sampler
=============================================================================
Multi-signal scoring + diversity-guaranteed frame selection.

Handles all incident types:
  - Moving entity (bear/dog crossing): tracks cluster centroid movement
  - Multi-entity (bus+scooter): detects cluster splits/merges
  - Dense cluster formation: measures concentration change (entropy drop)
  - Pedestrian hit-and-disappear: catches visual content change frames
  - Static aftermath: enforces minimum frame difference, removes duplicates

Pipeline:
  1. Dense scan (~100 frames) → compute per-frame signals
  2. Multi-signal importance scoring
  3. Key moment detection (onset, peak, content-change spikes)
  4. Diversity-guaranteed selection (no two frames > SIMILARITY_THRESHOLD)
  5. Save frames + heatmap pairs + analysis metadata

Output:
    OUTPUT_DIR/{video_id}/frame_00045.jpg
    OUTPUT_DIR/{video_id}_heatmap/frame_00045.jpg
    ANALYSIS_CSV with per-video heatmap features
=============================================================================
"""

import os
import cv2
import glob
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import label as ndlabel
from scipy.signal import savgol_filter
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                          CONFIGURATION                                   ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# ── Paths ──────────────────────────────────────────────────────────────────
VIDEOS_DIR       = "/kaggle/input/datasets/shresthml/vqa-autopilot/videos"
HEATMAPS_DIR     = "/kaggle/input/datasets/shresthml/vqa-autopilot/heatmaps"
OUTPUT_DIR       = "/kaggle/working/sampled_frames"
ANALYSIS_CSV     = "/kaggle/working/heatmap_analysis.csv"

# ── Dataset Sampling ───────────────────────────────────────────────────────
DATASET_PERCENT  = 100.0       # 5.0 for quick test, 100.0 for full run
RANDOM_SEED      = 42

# ── Visualization ──────────────────────────────────────────────────────────
SAMPLING_EXAMPLES = 10         # Show N random videos (0 = skip viz)
VIZ_OUTPUT_DIR    = "/kaggle/working/sampling_viz"

# ── Frame Sampling ─────────────────────────────────────────────────────────
NUM_OUTPUT_FRAMES    = 20      # Final frames to output per video
DENSE_SCAN_COUNT     = 40     # Frames to scan for signal computation
FRAME_SAVE_HEIGHT    = 480     # Resize height (None = original)
FRAME_SAVE_WIDTH     = None    # None = auto from aspect ratio

# ── Heatmap Analysis ──────────────────────────────────────────────────────
BRIGHTNESS_THRESHOLD   = 180   # Pixel value for "dot" detection
CLUSTER_MIN_AREA       = 40    # Min pixels for a valid cluster
CLUSTER_MERGE_KERNEL   = 35    # Morphological close kernel size
ONSET_GRADIENT_PCTILE  = 75    # Gradient percentile for onset detection

# ── Diversity & Dedup ──────────────────────────────────────────────────────
SIMILARITY_THRESHOLD   = 0.96  # Frames more similar than this = duplicate
MIN_FRAME_DIFFERENCE   = 0.04  # Minimum 4% pixel difference required
THUMB_SIZE             = 32    # Thumbnail size for similarity comparison
DIVERSITY_WINDOW       = 3     # Compare against this many nearest selected

# ── Importance Weights (tune these to change what "matters") ───────────────
W_CONTENT_CHANGE   = 1.0      # Weight for visual content difference
W_HEATMAP_ACTIVITY = 1.5      # Weight for heatmap dot coverage
W_CONCENTRATION    = 2.0      # Weight for cluster concentration change (entropy drop)
W_CLUSTER_MOTION   = 1.2      # Weight for cluster centroid movement
W_CLUSTER_BIRTH    = 2.5      # Weight for new cluster appearing
W_CONTENT_SPIKE    = 1.8      # Weight for sudden visual change (entity appears/disappears)

VERBOSE = True


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                    SIGNAL COMPUTATION (Per-Frame)                         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

class FrameSignals:
    """
    Compute multiple signals across densely-sampled frames.
    Each signal captures a different aspect of "something important happened."
    """

    def __init__(self, heatmap_path: str, video_path: str):
        self.heatmap_path = heatmap_path
        self.video_path = video_path

        # Will be populated by compute()
        self.frame_indices = np.array([])
        self.total_frames = 0
        self.fps = 0.0

        # Raw signals (one value per sampled frame)
        self.coverage = np.array([])          # heatmap bright pixel count
        self.concentration = np.array([])     # inverse spatial spread (high = dots clustered tight)
        self.num_clusters = np.array([])      # number of distinct clusters
        self.centroid_x = np.array([])        # main cluster centroid X
        self.centroid_y = np.array([])        # main cluster centroid Y
        self.content_diff = np.array([])      # frame-to-frame visual difference
        self.heatmap_diff = np.array([])      # frame-to-frame heatmap difference

        # Derived signals
        self.coverage_gradient = np.array([])
        self.concentration_gradient = np.array([])
        self.centroid_displacement = np.array([])
        self.cluster_birth = np.array([])     # new cluster appearing

        # Thumbnails for diversity checking
        self.video_thumbs = []                # list of (idx, thumb_array)

    def compute(self):
        """Run dense scan and compute all signals."""

        # ── Open both videos ───────────────────────────────────────────────
        cap_h = cv2.VideoCapture(self.heatmap_path)
        cap_v = cv2.VideoCapture(self.video_path)

        if not cap_h.isOpened() or not cap_v.isOpened():
            cap_h.release()
            cap_v.release()
            return False

        self.total_frames = int(cap_v.get(cv2.CAP_PROP_FRAME_COUNT))
        self.fps = cap_v.get(cv2.CAP_PROP_FPS)

        if self.total_frames <= 0:
            cap_h.release()
            cap_v.release()
            return False

        n_scan = min(DENSE_SCAN_COUNT, self.total_frames)
        self.frame_indices = np.linspace(0, self.total_frames - 1, n_scan, dtype=int)

        # ── Allocate signal arrays ─────────────────────────────────────────
        n = len(self.frame_indices)
        self.coverage = np.zeros(n)
        self.concentration = np.zeros(n)
        self.num_clusters = np.zeros(n, dtype=int)
        self.centroid_x = np.full(n, np.nan)
        self.centroid_y = np.full(n, np.nan)
        self.content_diff = np.zeros(n)
        self.heatmap_diff = np.zeros(n)

        prev_video_gray = None
        prev_heatmap_gray = None

        for i, idx in enumerate(self.frame_indices):
            # ── Read heatmap frame ─────────────────────────────────────────
            cap_h.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret_h, frame_h = cap_h.read()

            if ret_h and frame_h is not None:
                gray_h = cv2.cvtColor(frame_h, cv2.COLOR_BGR2GRAY)
                bright = (gray_h > BRIGHTNESS_THRESHOLD).astype(np.uint8) * 255

                # Coverage: total bright pixels
                self.coverage[i] = float(bright.sum() / 255.0)

                # Concentration: how tightly clustered are the dots?
                # Low std dev of dot positions = high concentration
                if self.coverage[i] > CLUSTER_MIN_AREA:
                    ys, xs = np.where(bright > 0)
                    std_x = np.std(xs)
                    std_y = np.std(ys)
                    spread = std_x + std_y
                    # Invert: high spread → low concentration
                    self.concentration[i] = 1.0 / max(spread, 1.0) * 1000

                    # Main centroid
                    self.centroid_x[i] = float(np.mean(xs))
                    self.centroid_y[i] = float(np.mean(ys))

                    # Cluster count via connected components
                    kernel = cv2.getStructuringElement(
                        cv2.MORPH_ELLIPSE,
                        (CLUSTER_MERGE_KERNEL, CLUSTER_MERGE_KERNEL)
                    )
                    closed = cv2.morphologyEx(bright, cv2.MORPH_CLOSE, kernel)
                    labeled, n_feat = ndlabel(closed)
                    # Filter small clusters
                    valid_clusters = 0
                    for cid in range(1, n_feat + 1):
                        if (labeled == cid).sum() >= CLUSTER_MIN_AREA:
                            valid_clusters += 1
                    self.num_clusters[i] = valid_clusters

                # Heatmap frame-to-frame diff
                if prev_heatmap_gray is not None:
                    diff = cv2.absdiff(gray_h, prev_heatmap_gray)
                    self.heatmap_diff[i] = float(diff.mean())
                prev_heatmap_gray = gray_h.copy()

            # ── Read video frame ───────────────────────────────────────────
            cap_v.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret_v, frame_v = cap_v.read()

            if ret_v and frame_v is not None:
                gray_v = cv2.cvtColor(frame_v, cv2.COLOR_BGR2GRAY)

                # Content frame-to-frame diff
                if prev_video_gray is not None:
                    diff = cv2.absdiff(gray_v, prev_video_gray)
                    self.content_diff[i] = float(diff.mean())

                prev_video_gray = gray_v.copy()

                # Store thumbnail for diversity checking later
                thumb = cv2.resize(gray_v, (THUMB_SIZE, THUMB_SIZE)).flatten().astype(np.float32)
                norm = np.linalg.norm(thumb)
                if norm > 0:
                    thumb /= norm
                self.video_thumbs.append((int(idx), thumb))

        cap_h.release()
        cap_v.release()

        # ── Compute derived signals ────────────────────────────────────────
        self._compute_derived()

        return True

    def _compute_derived(self):
        """Compute gradients, displacements, and cluster birth signals."""
        n = len(self.frame_indices)

        # Smooth coverage before gradient (reduces noise)
        if n > 7:
            win = min(7, n if n % 2 == 1 else n - 1)
            smoothed_cov = savgol_filter(self.coverage, win, 2)
        else:
            smoothed_cov = self.coverage.copy()

        self.coverage_gradient = np.abs(np.gradient(smoothed_cov))
        self.concentration_gradient = np.abs(np.gradient(self.concentration))

        # Centroid displacement (frame-to-frame)
        self.centroid_displacement = np.zeros(n)
        for i in range(1, n):
            if not np.isnan(self.centroid_x[i]) and not np.isnan(self.centroid_x[i - 1]):
                dx = self.centroid_x[i] - self.centroid_x[i - 1]
                dy = self.centroid_y[i] - self.centroid_y[i - 1]
                self.centroid_displacement[i] = np.sqrt(dx**2 + dy**2)

        # Cluster birth: cluster count increased
        self.cluster_birth = np.zeros(n)
        for i in range(1, n):
            delta = self.num_clusters[i] - self.num_clusters[i - 1]
            if delta > 0:
                self.cluster_birth[i] = float(delta) * 100  # scale up


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                   IMPORTANCE SCORING & KEY MOMENTS                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def _safe_normalize(arr: np.ndarray) -> np.ndarray:
    """Normalize to [0, 1]. Returns zeros if max == 0."""
    mx = arr.max()
    if mx > 0:
        return arr / mx
    return np.zeros_like(arr)


def compute_importance_scores(signals: FrameSignals) -> np.ndarray:
    """
    Combine all signals into a single importance score per frame.
    Higher score = more important frame to keep.
    """
    n = len(signals.frame_indices)
    if n == 0:
        return np.array([])

    # Normalize each signal to [0, 1]
    s_content    = _safe_normalize(signals.content_diff)
    s_coverage   = _safe_normalize(signals.coverage)
    s_conc       = _safe_normalize(signals.concentration_gradient)
    s_motion     = _safe_normalize(signals.centroid_displacement)
    s_birth      = _safe_normalize(signals.cluster_birth)
    s_cov_grad   = _safe_normalize(signals.coverage_gradient)

    # Content spike: frames where visual content changes dramatically
    # (entity appears, disappears, collision deformation, etc.)
    content_mean = s_content.mean()
    content_std = s_content.std()
    s_spike = np.zeros(n)
    if content_std > 0:
        s_spike = np.maximum(0, (s_content - content_mean) / content_std)
        s_spike = _safe_normalize(s_spike)

    # Weighted sum
    scores = (
        W_CONTENT_CHANGE   * s_content +
        W_HEATMAP_ACTIVITY * s_coverage +
        W_CONCENTRATION    * s_conc +
        W_CLUSTER_MOTION   * s_motion +
        W_CLUSTER_BIRTH    * s_birth +
        W_CONTENT_SPIKE    * s_spike +
        W_HEATMAP_ACTIVITY * s_cov_grad * 0.5  # bonus for coverage gradient
    )

    return scores


def find_key_moments(signals: FrameSignals, scores: np.ndarray) -> dict:
    """
    Detect key moments in the video timeline:
      - onset: where heatmap activity begins rising significantly
      - peak: maximum heatmap coverage
      - content_peaks: frames with large visual content change
      - cluster_events: frames where clusters appear/split
    """
    n = len(signals.frame_indices)
    if n == 0:
        return {"onset_idx": 0, "peak_idx": 0, "aftermath_idx": 0,
                "content_peaks": [], "cluster_events": []}

    # Peak = max coverage
    peak_idx = int(np.argmax(signals.coverage))

    # Onset = first frame where coverage gradient exceeds threshold
    grad = signals.coverage_gradient[:peak_idx + 1] if peak_idx > 0 else signals.coverage_gradient
    if len(grad) > 0 and grad.max() > 0:
        threshold = np.percentile(grad[grad > 0], ONSET_GRADIENT_PCTILE) \
            if np.any(grad > 0) else 0
        candidates = np.where(grad >= max(threshold, 1e-6))[0]
        onset_idx = int(candidates[0]) if len(candidates) > 0 else max(0, peak_idx - n // 10)
    else:
        onset_idx = max(0, peak_idx - n // 10)

    # Aftermath = coverage drops below 50% of peak
    post_peak = signals.coverage[peak_idx:]
    half_peak = signals.coverage[peak_idx] * 0.5
    aftermath_cands = np.where(post_peak < half_peak)[0]
    aftermath_idx = peak_idx + int(aftermath_cands[0]) if len(aftermath_cands) > 0 \
        else min(n - 1, peak_idx + n // 10)

    # Content peaks: top frames by content_diff (excluding near-zero)
    content_sorted = np.argsort(signals.content_diff)[::-1]
    content_peaks = []
    for ci in content_sorted[:10]:
        if signals.content_diff[ci] > signals.content_diff.mean() * 1.5:
            content_peaks.append(int(ci))

    # Cluster events: frames where cluster count changed
    cluster_events = []
    for i in range(1, n):
        if signals.num_clusters[i] != signals.num_clusters[i - 1]:
            cluster_events.append(int(i))

    return {
        "onset_idx": onset_idx,
        "peak_idx": peak_idx,
        "aftermath_idx": aftermath_idx,
        "content_peaks": content_peaks,
        "cluster_events": cluster_events,
    }


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║              DIVERSITY-GUARANTEED FRAME SELECTION                         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two thumbnail vectors."""
    return float(np.dot(a, b))


def pixel_difference(a: np.ndarray, b: np.ndarray) -> float:
    """Mean absolute difference between two thumbnail vectors (pre-normalized)."""
    return float(np.mean(np.abs(a - b)))


def select_diverse_frames(signals: FrameSignals, scores: np.ndarray,
                          key_moments: dict) -> np.ndarray:
    """
    Select NUM_OUTPUT_FRAMES frames that are:
      1. High importance score
      2. Sufficiently different from each other (diversity)
      3. Cover key moments (onset, peak, aftermath guaranteed)
      4. Temporally spread (not all from same 1-second window)

    Algorithm:
      1. Force-include onset, peak, aftermath, top content-change frames
      2. Sort remaining by importance score
      3. Greedily add frames: skip if too similar to ANY already-selected frame
      4. If still short, relax threshold and retry
    """
    n = len(signals.frame_indices)
    if n <= NUM_OUTPUT_FRAMES:
        return signals.frame_indices.copy()

    thumb_map = {idx: thumb for idx, thumb in signals.video_thumbs}

    def _get_thumb(scan_idx: int):
        """Get thumbnail for scan index → actual frame index."""
        actual_idx = int(signals.frame_indices[scan_idx])
        return thumb_map.get(actual_idx, None)

    # ── Step 1: Force-include critical frames ──────────────────────────────
    must_include = set()
    must_include.add(key_moments["onset_idx"])
    must_include.add(key_moments["peak_idx"])
    must_include.add(key_moments["aftermath_idx"])

    # Add top 2 content-change frames
    for ci in key_moments["content_peaks"][:2]:
        must_include.add(ci)

    # Add first cluster event (entity appears)
    for ci in key_moments["cluster_events"][:1]:
        must_include.add(ci)

    # Clamp to valid range
    must_include = {max(0, min(i, n - 1)) for i in must_include}

    # ── Step 2: Greedy diversity selection ──────────────────────────────────
    selected_scan_indices = sorted(must_include)

    # Remove must-includes from candidate pool, sort rest by score descending
    candidates = [(i, scores[i]) for i in range(n) if i not in must_include]
    candidates.sort(key=lambda x: x[1], reverse=True)

    threshold = SIMILARITY_THRESHOLD

    for attempt in range(3):  # relax threshold up to 2 times
        for cand_idx, cand_score in candidates:
            if len(selected_scan_indices) >= NUM_OUTPUT_FRAMES:
                break

            if cand_idx in selected_scan_indices:
                continue

            # Check diversity against already-selected frames
            cand_thumb = _get_thumb(cand_idx)
            if cand_thumb is None:
                continue

            too_similar = False
            for sel_idx in selected_scan_indices:
                sel_thumb = _get_thumb(sel_idx)
                if sel_thumb is None:
                    continue

                sim = cosine_sim(cand_thumb, sel_thumb)
                if sim > threshold:
                    too_similar = True
                    break

            if not too_similar:
                selected_scan_indices.append(cand_idx)
                selected_scan_indices.sort()

        if len(selected_scan_indices) >= NUM_OUTPUT_FRAMES:
            break

        # Relax threshold for next attempt
        threshold += 0.015

    # ── Step 3: If still short, fill with evenly-spaced frames ─────────────
    if len(selected_scan_indices) < NUM_OUTPUT_FRAMES:
        remaining_needed = NUM_OUTPUT_FRAMES - len(selected_scan_indices)
        uniform = np.linspace(0, n - 1, remaining_needed + len(selected_scan_indices),
                              dtype=int)
        for u in uniform:
            if len(selected_scan_indices) >= NUM_OUTPUT_FRAMES:
                break
            if u not in selected_scan_indices:
                selected_scan_indices.append(int(u))
        selected_scan_indices.sort()

    # Cap and convert to actual frame indices
    selected_scan_indices = sorted(set(selected_scan_indices))[:NUM_OUTPUT_FRAMES]
    actual_indices = np.array([int(signals.frame_indices[i])
                               for i in selected_scan_indices])

    return actual_indices


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                   PEAK FRAME SPATIAL ANALYSIS                            ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def analyze_peak_clusters(heatmap_path: str, peak_frame_idx: int) -> dict:
    """
    At the peak frame, extract spatial features from heatmap clusters.
    """
    result = {
        "num_clusters": 0, "clusters": [],
        "brightest_point": (0, 0), "brightest_region": "center",
        "frame_width": 0, "frame_height": 0,
    }

    cap = cv2.VideoCapture(heatmap_path)
    if not cap.isOpened():
        return result

    cap.set(cv2.CAP_PROP_POS_FRAMES, peak_frame_idx)
    ret, frame = cap.read()
    cap.release()

    if not ret or frame is None:
        return result

    h, w = frame.shape[:2]
    result["frame_width"] = w
    result["frame_height"] = h

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, BRIGHTNESS_THRESHOLD, 255, cv2.THRESH_BINARY)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (CLUSTER_MERGE_KERNEL, CLUSTER_MERGE_KERNEL))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    labeled, n_feat = ndlabel(closed)

    clusters = []
    for cid in range(1, n_feat + 1):
        mask = (labeled == cid)
        area = int(mask.sum())
        if area < CLUSTER_MIN_AREA:
            continue

        ys, xs = np.where(mask)
        cx, cy = float(xs.mean()), float(ys.mean())
        rel_x, rel_y = cx / w, cy / h

        h_pos = "left" if rel_x < 0.33 else ("right" if rel_x > 0.67 else "center")
        v_pos = "top" if rel_y < 0.4 else ("bottom" if rel_y > 0.7 else "middle")

        area_ratio = area / (w * h)
        size_class = "large" if area_ratio > 0.05 else ("medium" if area_ratio > 0.01 else "small")

        clusters.append({
            "area": area, "area_ratio": round(area_ratio, 5),
            "centroid": (round(cx, 1), round(cy, 1)),
            "h_position": h_pos, "v_position": v_pos,
            "size_class": size_class,
            "brightness": round(float(gray[mask].mean()), 1),
            "rel_x": round(rel_x, 3), "rel_y": round(rel_y, 3),
        })

    clusters.sort(key=lambda c: c["area"], reverse=True)

    bp = np.unravel_index(np.argmax(gray), gray.shape)
    bp_rel_x = int(bp[1]) / w
    bp_region = "left" if bp_rel_x < 0.33 else ("right" if bp_rel_x > 0.67 else "center")

    result["num_clusters"] = len(clusters)
    result["clusters"] = clusters
    result["brightest_point"] = (int(bp[1]), int(bp[0]))
    result["brightest_region"] = bp_region
    return result


def analyze_motion_direction(signals: FrameSignals, key_moments: dict) -> dict:
    """Summarize cluster motion between onset and peak."""
    onset = key_moments["onset_idx"]
    peak = key_moments["peak_idx"]

    if peak <= onset or onset >= len(signals.centroid_x):
        return {"direction": "unknown", "speed": 0.0}

    # Get non-NaN centroids in the onset→peak window
    window_cx = signals.centroid_x[onset:peak + 1]
    window_cy = signals.centroid_y[onset:peak + 1]
    valid = ~(np.isnan(window_cx) | np.isnan(window_cy))

    if valid.sum() < 2:
        return {"direction": "unknown", "speed": 0.0}

    cx_valid = window_cx[valid]
    cy_valid = window_cy[valid]

    dx = cx_valid[-1] - cx_valid[0]
    dy = cy_valid[-1] - cy_valid[0]
    dist = np.sqrt(dx**2 + dy**2)
    n_frames = int(valid.sum())
    speed = dist / max(n_frames, 1)

    if dist < 10:
        direction = "stationary"
    elif abs(dx) > abs(dy) * 1.5:
        direction = "left-to-right" if dx > 0 else "right-to-left"
    elif dy > 0:
        direction = "approaching"
    else:
        direction = "receding"

    return {"direction": direction, "speed": round(speed, 2)}


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                       FRAME EXTRACTION & SAVE                            ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def extract_and_save(video_path: str, heatmap_path: str,
                     frame_indices: np.ndarray,
                     vid_out: str, hm_out: str):
    """Save selected frames from both video and heatmap."""
    os.makedirs(vid_out, exist_ok=True)
    os.makedirs(hm_out, exist_ok=True)

    cap_v = cv2.VideoCapture(video_path)
    cap_h = cv2.VideoCapture(heatmap_path)

    for idx in frame_indices:
        fname = f"frame_{int(idx):05d}.jpg"

        for cap, out_dir in [(cap_v, vid_out), (cap_h, hm_out)]:
            if not cap.isOpened():
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if ret and frame is not None:
                if FRAME_SAVE_HEIGHT is not None:
                    fh, fw = frame.shape[:2]
                    tw = FRAME_SAVE_WIDTH or max(1, int(fw * FRAME_SAVE_HEIGHT / fh))
                    frame = cv2.resize(frame, (tw, FRAME_SAVE_HEIGHT))
                cv2.imwrite(os.path.join(out_dir, fname), frame)

    cap_v.release()
    cap_h.release()


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                       TEXT SUMMARY                                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def build_summary(signals: FrameSignals, key_moments: dict,
                  peak_info: dict, motion: dict) -> str:
    """Build structured text summary for VLM consumption."""
    lines = []

    onset_f = int(signals.frame_indices[key_moments["onset_idx"]])
    peak_f = int(signals.frame_indices[key_moments["peak_idx"]])
    after_f = int(signals.frame_indices[key_moments["aftermath_idx"]])

    lines.append(f"Incident timeline: onset=frame {onset_f}, peak=frame {peak_f}, "
                 f"aftermath=frame {after_f} (of {signals.total_frames} total).")

    nc = peak_info["num_clusters"]
    if nc == 0:
        lines.append("No heatmap clusters detected at peak.")
    elif nc == 1:
        c = peak_info["clusters"][0]
        lines.append(f"1 entity cluster: {c['size_class']}, "
                     f"{c['h_position']}-{c['v_position']}.")
    else:
        lines.append(f"{nc} entity clusters at peak:")
        for i, c in enumerate(peak_info["clusters"][:4]):
            tag = "Primary" if i == 0 else f"Entity {i+1}"
            lines.append(f"  {tag}: {c['size_class']}, {c['h_position']}-{c['v_position']}, "
                         f"brightness={c['brightness']:.0f}")

    lines.append(f"Brightest region: {peak_info['brightest_region']}.")
    lines.append(f"Pre-incident motion: {motion['direction']} (speed={motion['speed']:.1f}).")

    n_content = len(key_moments["content_peaks"])
    n_cluster = len(key_moments["cluster_events"])
    if n_content > 0:
        lines.append(f"Detected {n_content} significant visual content changes.")
    if n_cluster > 0:
        lines.append(f"Detected {n_cluster} cluster appearance/split events.")

    return "\n".join(lines)


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                       VISUALIZATION                                      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def visualize(video_id: int, video_path: str, heatmap_path: str,
              frame_indices: np.ndarray, signals: FrameSignals,
              scores: np.ndarray, key_moments: dict, viz_dir: str) -> str:
    """
    4-row visualization:
      Row 1: Selected video frames
      Row 2: Corresponding heatmap frames
      Row 3: Multi-signal importance plot with selected frames marked
      Row 4: Individual signal breakdown
    """
    n_show = min(len(frame_indices), 6)
    if n_show < len(frame_indices):
        pick = np.linspace(0, len(frame_indices) - 1, n_show, dtype=int)
        show_frames = frame_indices[pick]
    else:
        show_frames = frame_indices

    fig = plt.figure(figsize=(3.5 * n_show, 14))

    # ── Row 1 & 2: Video + Heatmap frames ─────────────────────────────────
    cap_v = cv2.VideoCapture(video_path)
    cap_h = cv2.VideoCapture(heatmap_path)

    for col, fidx in enumerate(show_frames):
        # Video frame
        ax = fig.add_subplot(4, n_show, col + 1)
        cap_v.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
        ret, f = cap_v.read()
        if ret:
            ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        ax.set_title(f"f{int(fidx)}", fontsize=8)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel("Video", fontsize=10, fontweight="bold")

        # Heatmap frame
        ax = fig.add_subplot(4, n_show, n_show + col + 1)
        cap_h.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
        ret, f = cap_h.read()
        if ret:
            ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        ax.set_title(f"f{int(fidx)}", fontsize=8)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel("Heatmap", fontsize=10, fontweight="bold")

    cap_v.release()
    cap_h.release()

    # ── Row 3: Combined importance score ───────────────────────────────────
    ax3 = fig.add_subplot(4, 1, 3)
    x = signals.frame_indices
    ax3.fill_between(x, scores, alpha=0.3, color="#2196F3")
    ax3.plot(x, scores, color="#2196F3", linewidth=1.5, label="Importance")

    onset_f = int(signals.frame_indices[key_moments["onset_idx"]])
    peak_f = int(signals.frame_indices[key_moments["peak_idx"]])
    after_f = int(signals.frame_indices[key_moments["aftermath_idx"]])

    ax3.axvline(onset_f, color="orange", ls="--", lw=1.5, label="Onset")
    ax3.axvline(peak_f, color="red", ls="-", lw=2, label="Peak")
    ax3.axvline(after_f, color="green", ls="--", lw=1.5, label="Aftermath")

    for fi in frame_indices:
        ax3.axvline(fi, color="gray", alpha=0.4, lw=0.5)
    ax3.scatter(frame_indices, np.interp(frame_indices, x, scores),
                color="red", s=30, zorder=5, label="Selected")

    ax3.set_ylabel("Importance")
    ax3.set_title(f"Video {video_id} — Importance Score", fontsize=10)
    ax3.legend(fontsize=7, loc="upper left")

    # ── Row 4: Signal breakdown ────────────────────────────────────────────
    ax4 = fig.add_subplot(4, 1, 4)
    ax4.plot(x, _safe_normalize(signals.coverage), label="Coverage", alpha=0.8)
    ax4.plot(x, _safe_normalize(signals.concentration), label="Concentration", alpha=0.8)
    ax4.plot(x, _safe_normalize(signals.content_diff), label="Content Δ", alpha=0.8)
    ax4.plot(x, _safe_normalize(signals.centroid_displacement), label="Motion", alpha=0.8)
    ax4.plot(x, _safe_normalize(signals.cluster_birth), label="Cluster birth", alpha=0.8)

    ax4.axvline(onset_f, color="orange", ls="--", lw=1, alpha=0.5)
    ax4.axvline(peak_f, color="red", ls="-", lw=1, alpha=0.5)

    ax4.set_xlabel("Frame index")
    ax4.set_ylabel("Normalized signal")
    ax4.set_title("Signal Breakdown", fontsize=10)
    ax4.legend(fontsize=7, loc="upper left", ncol=3)

    plt.suptitle(f"Video {video_id} — Smart Sampler Analysis", fontsize=12, fontweight="bold")
    plt.tight_layout()

    os.makedirs(viz_dir, exist_ok=True)
    out = os.path.join(viz_dir, f"sampling_v{video_id}.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.close()
    return out


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                              MAIN                                        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def main():
    t0 = time.time()
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── Discover videos ────────────────────────────────────────────────────
    video_files = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.mp4")))
    all_videos = []
    for vp in video_files:
        try:
            vid = int(Path(vp).stem)
            hp = os.path.join(HEATMAPS_DIR, f"{vid}.mp4")
            all_videos.append((vid, vp, hp if os.path.exists(hp) else None))
        except ValueError:
            pass

    all_videos.sort(key=lambda x: x[0])
    print(f"Videos found: {len(all_videos)}")

    # ── Apply DATASET_PERCENT ──────────────────────────────────────────────
    np.random.seed(RANDOM_SEED)
    n_use = max(1, int(len(all_videos) * DATASET_PERCENT / 100.0))
    if n_use < len(all_videos):
        chosen = sorted(np.random.choice(len(all_videos), n_use, replace=False))
        all_videos = [all_videos[i] for i in chosen]
    print(f"Processing: {len(all_videos)} ({DATASET_PERCENT}%)\n")

    # ── Viz selection ──────────────────────────────────────────────────────
    viz_ids = set()
    if SAMPLING_EXAMPLES > 0:
        viz_n = min(SAMPLING_EXAMPLES, len(all_videos))
        viz_ids = {all_videos[i][0]
                   for i in np.random.choice(len(all_videos), viz_n, replace=False)}
        print(f"Visualizing: {sorted(viz_ids)}\n")

    # ── Process ────────────────────────────────────────────────────────────
    rows = []

    for i, (vid, vpath, hpath) in enumerate(tqdm(all_videos, desc="Sampling")):

        # ── Compute signals ────────────────────────────────────────────────
        signals = FrameSignals(hpath or vpath, vpath)
        ok = signals.compute()

        if not ok:
            # Fallback: uniform sampling
            cap = cv2.VideoCapture(vpath)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            indices = np.linspace(0, max(total - 1, 0), NUM_OUTPUT_FRAMES, dtype=int)
            vid_out = os.path.join(OUTPUT_DIR, str(vid))
            hm_out = os.path.join(OUTPUT_DIR, f"{vid}_heatmap")
            extract_and_save(vpath, hpath or vpath, indices, vid_out, hm_out)

            rows.append({"video_id": vid, "total_frames": total,
                         "status": "fallback_uniform",
                         "n_sampled": len(indices),
                         "sampled_indices": ",".join(map(str, indices.tolist())),
                         "summary": "Fallback: uniform sampling (signal computation failed)"})
            continue

        # ── Score & select ─────────────────────────────────────────────────
        scores = compute_importance_scores(signals)
        key_moments = find_key_moments(signals, scores)
        selected = select_diverse_frames(signals, scores, key_moments)

        # ── Save frames ────────────────────────────────────────────────────
        vid_out = os.path.join(OUTPUT_DIR, str(vid))
        hm_out = os.path.join(OUTPUT_DIR, f"{vid}_heatmap")
        extract_and_save(vpath, hpath or vpath, selected, vid_out, hm_out)

        # ── Spatial analysis ───────────────────────────────────────────────
        peak_actual = int(signals.frame_indices[key_moments["peak_idx"]])
        peak_info = analyze_peak_clusters(hpath or vpath, peak_actual)
        motion = analyze_motion_direction(signals, key_moments)
        summary = build_summary(signals, key_moments, peak_info, motion)

        # ── Build row ──────────────────────────────────────────────────────
        row = {
            "video_id": vid,
            "total_frames": signals.total_frames,
            "fps": round(signals.fps, 2),
            "status": "ok",
            "onset_frame": int(signals.frame_indices[key_moments["onset_idx"]]),
            "peak_frame": peak_actual,
            "aftermath_frame": int(signals.frame_indices[key_moments["aftermath_idx"]]),
            "peak_position": round(peak_actual / max(signals.total_frames - 1, 1), 3),
            "num_clusters": peak_info["num_clusters"],
            "brightest_region": peak_info["brightest_region"],
            "motion_direction": motion["direction"],
            "motion_speed": motion["speed"],
            "n_content_changes": len(key_moments["content_peaks"]),
            "n_cluster_events": len(key_moments["cluster_events"]),
            "n_sampled": len(selected),
            "sampled_indices": ",".join(map(str, selected.tolist())),
            "summary": summary,
        }

        for ci in range(3):
            if ci < len(peak_info["clusters"]):
                c = peak_info["clusters"][ci]
                row[f"c{ci}_size"] = c["size_class"]
                row[f"c{ci}_hpos"] = c["h_position"]
                row[f"c{ci}_vpos"] = c["v_position"]
                row[f"c{ci}_area"] = c["area_ratio"]
                row[f"c{ci}_bright"] = c["brightness"]
            else:
                row[f"c{ci}_size"] = ""
                row[f"c{ci}_hpos"] = ""
                row[f"c{ci}_vpos"] = ""
                row[f"c{ci}_area"] = 0
                row[f"c{ci}_bright"] = 0

        rows.append(row)

        # ── Visualize ──────────────────────────────────────────────────────
        if vid in viz_ids:
            vp = visualize(vid, vpath, hpath or vpath,
                           selected, signals, scores, key_moments, VIZ_OUTPUT_DIR)
            if VERBOSE:
                print(f"  Viz: {vp}")

        if VERBOSE and (i + 1) % 100 == 0:
            el = time.time() - t0
            print(f"  [{i+1}/{len(all_videos)}] {el:.0f}s elapsed")

    # ── Save CSV ───────────────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    df.to_csv(ANALYSIS_CSV, index=False)

    elapsed = time.time() - t0
    print(f"\n{'='*60}")
    print(f"  DONE — {len(all_videos)} videos in {elapsed:.1f}s")
    print(f"  Output frames : {OUTPUT_DIR}")
    print(f"  Analysis CSV  : {ANALYSIS_CSV}")
    if SAMPLING_EXAMPLES > 0:
        print(f"  Visualizations: {VIZ_OUTPUT_DIR}")
    print(f"{'='*60}")

    # ── Quick stats ────────────────────────────────────────────────────────
    ok_df = df[df["status"] == "ok"]
    if len(ok_df) > 0:
        print(f"\n── Analysis Stats ({len(ok_df)} videos) ──")
        print(f"  Clusters:  {ok_df['num_clusters'].value_counts().sort_index().to_dict()}")
        print(f"  Bright region: {ok_df['brightest_region'].value_counts().to_dict()}")
        print(f"  Motion: {ok_df['motion_direction'].value_counts().to_dict()}")
        print(f"  Peak position: mean={ok_df['peak_position'].mean():.2f} std={ok_df['peak_position'].std():.2f}")
        print(f"  Content changes: mean={ok_df['n_content_changes'].mean():.1f}")
        print(f"  Cluster events: mean={ok_df['n_cluster_events'].mean():.1f}")

    return df


if __name__ == "__main__":
    df_analysis = main()

Videos found: 661
Processing: 661 (100.0%)

Visualizing: [72, 90, 135, 247, 271, 305, 483, 543, 624, 630]



Sampling:   0%|          | 0/661 [00:00<?, ?it/s]

  Viz: /kaggle/working/sampling_viz/sampling_v72.png
  Viz: /kaggle/working/sampling_viz/sampling_v90.png
  [100/661] 1325s elapsed
  Viz: /kaggle/working/sampling_viz/sampling_v135.png
  [200/661] 2599s elapsed
  Viz: /kaggle/working/sampling_viz/sampling_v247.png
  Viz: /kaggle/working/sampling_viz/sampling_v271.png
  [300/661] 3904s elapsed
  Viz: /kaggle/working/sampling_viz/sampling_v305.png
  [400/661] 5220s elapsed
  Viz: /kaggle/working/sampling_viz/sampling_v483.png
  [500/661] 6475s elapsed
  Viz: /kaggle/working/sampling_viz/sampling_v543.png
  [600/661] 7805s elapsed
  Viz: /kaggle/working/sampling_viz/sampling_v624.png
  Viz: /kaggle/working/sampling_viz/sampling_v630.png

  DONE — 661 videos in 8589.7s
  Output frames : /kaggle/working/sampled_frames
  Analysis CSV  : /kaggle/working/heatmap_analysis.csv
  Visualizations: /kaggle/working/sampling_viz

── Analysis Stats (661 videos) ──
  Clusters:  {1: 103, 2: 77, 3: 50, 4: 34, 5: 37, 6: 29, 7: 15, 8: 15, 9: 18, 10: 15, 11